# 02 — BERT: andamento e risultati

BERT viene addestrato sul server `artes4` (script `02_bert.py`), sulla versione **min0** con i **fold per gruppo**:
gli stessi dati e gli stessi fold del classificatore a feature (notebook 01).
Questo notebook copia dal server il log e i fold già finiti, mostra a che punto è l'addestramento e
confronta BERT con il classificatore a feature **sugli stessi fold**, livello per livello.

**Prima di eseguirlo** apri dal terminale la connessione al server (chiede la password una volta, poi resta aperta in background):

```bash
ssh -M -S ~/.ssh/artes4.sock -fN fabio.pinelli@10.181.100.6
```

Il notebook si può rieseguire quante volte si vuole: ogni volta riprende i fold nuovi.

In [1]:
import re
import subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from comune import RESULTS, LIVELLI, per_testata, tabella_per_fold

SERVER = "fabio.pinelli@10.181.100.6"
SOCKET = str(Path.home() / ".ssh" / "artes4.sock")
REMOTO = "~/John/WebScience2027/Results/2-classificazione/"
LOCALE = Path(RESULTS)
VERSIONE, SCHEMA = "min0", "fold_gruppo"
EPOCHE, FOLD = 5, 5

## 1. Copia dal server

In [2]:
(LOCALE / "bert").mkdir(exist_ok=True)
for sorgente, destinazione in [(REMOTO + "avanzamento_bert.log", LOCALE), (REMOTO + "bert/*.csv", LOCALE / "bert")]:
    esito = subprocess.run(["scp", "-o", f"ControlPath={SOCKET}", "-o", "BatchMode=yes", f"{SERVER}:{sorgente}", str(destinazione)],
                           capture_output=True, text=True)
    if esito.returncode == 0:
        print(f"copiato: {sorgente}")
    elif "No such file" in esito.stderr:
        print(f"ancora niente sul server: {sorgente}")
    else:
        print(f"ERRORE (la connessione è aperta? vedi in alto): {esito.stderr.strip()}")

## 2. A che punto è

In [3]:
log = [r for r in (LOCALE / "avanzamento_bert.log").read_text().splitlines() if f"{VERSIONE} {SCHEMA}" in r]
print("\n".join(log[-10:]))

epoche = pd.DataFrame([
    {"fold": int(m[1]), "epoca": int(m[2]), "secondi": int(m[3]), "loss": float(m[4])}
    for m in (re.search(r"fold (\d+): epoca (\d+)/\d+ finita in (\d+) s, loss media ([\d.]+)", r) for r in log) if m])

fatte = len(epoche)
print(f"\nepoche finite: {fatte} su {EPOCHE * FOLD}")
if fatte:
    restanti = (EPOCHE * FOLD - fatte) * epoche.secondi.mean()
    print(f"durata media di un'epoca: {epoche.secondi.mean() / 60:.0f} min -> tempo stimato che manca: {restanti / 3600:.1f} ore")

2026-10-13 14:25:22  min0 fold_gruppo fold 1: inizio (12683 articoli di training, 3587 di test)
2026-10-13 14:26:14  min0 fold_gruppo fold 1: epoca 1/5, passo 100/397, loss 1.3422

epoche finite: 0 su 25


In [4]:
if fatte:
    plt.figure(figsize=(6, 3.5))
    for fold, g in epoche.groupby("fold"):
        plt.plot(g.epoca, g.loss, marker="o", label=f"fold {fold}")
    plt.xticks(range(1, EPOCHE + 1))
    plt.xlabel("epoca")
    plt.ylabel("loss media sul training")
    plt.legend(fontsize=8)
    plt.show()

## 3. BERT contro il classificatore a feature, sui fold già finiti

Per un confronto alla pari, il classificatore a feature viene valutato **solo sugli stessi fold** che BERT ha già completato.

In [5]:
pronti = sorted((LOCALE / "bert").glob(f"{VERSIONE}_{SCHEMA}_fold*.csv"))
bert = pd.concat([pd.read_csv(f) for f in pronti]) if pronti else None
tfidf_tutto = pd.read_csv(LOCALE / f"01_tfidf_{VERSIONE}_{SCHEMA}_articoli.csv")
print("fold di BERT pronti:", sorted(bert.fold.unique()) if bert is not None else "nessuno")

fold di BERT pronti: nessuno


In [6]:
if bert is not None:
    tfidf = tfidf_tutto[tfidf_tutto.fold.isin(bert.fold.unique())]
    righe = []
    for modello, art in [("TF-IDF + LR", tfidf), ("BERT", bert)]:
        righe.append({"modello": modello, "livello": "articoli", **tabella_per_fold(art).loc["media"]})
        righe.append({"modello": modello, "livello": "testate", **tabella_per_fold(per_testata(art)).loc["media"]})
    display(pd.DataFrame(righe).set_index(["livello", "modello"]).sort_index())

Accuratezza per testata, livello per livello, separando testate **nei gruppi** e **indipendenti** (tra parentesi il numero di testate).

In [7]:
if bert is not None:
    dimensione = tfidf_tutto.groupby("domain").gruppo.first().pipe(lambda g: g.map(g.value_counts()))
    righe = []
    for modello, art in [("TF-IDF + LR", tfidf), ("BERT", bert)]:
        pub = per_testata(art)
        pub["tipo"] = pub.domain.map(dimensione).ge(2).map({True: "nei gruppi", False: "indipendenti"})
        pub["giusta"] = pub.y_true == pub.y_pred
        for (liv, tipo), g in pub.groupby(["y_true", "tipo"]):
            righe.append({"livello": LIVELLI[liv], "testate": tipo, "modello": modello, "valore": f"{g.giusta.mean():.2f} ({len(g)})"})
        for tipo, g in pub.groupby("tipo"):
            righe.append({"livello": "tutti", "testate": tipo, "modello": modello, "valore": f"{g.giusta.mean():.2f} ({len(g)})"})
    display(pd.DataFrame(righe).pivot(index=["livello", "testate"], columns="modello", values="valore")[["TF-IDF + LR", "BERT"]])